## Quality Assessment Visualization

This notebook visualizes the output of the Quality Assessment (QA) framework
produced by notebook `04_qa_framework`. It loads the QA NetCDF files for each
bias correction method (LS, LSEQM, LSEQM+DL) and creates a suite of
diagnostic plots to compare quality across methods and spatial domains.

### Visualizations provided

1. **CQI Spatial Maps** -- Continuous Quality Index for each method side by side.
2. **Categorical Quality Maps** -- Poor / Fair / Good / Excellent classification.
3. **Method Improvement Map** -- Quality gain from LS to LSEQM+DL.
4. **Component Quality Maps** -- Basic statistical, distribution, and temporal scores.
5. **Confidence Map** -- Spatial reliability of the quality assessment.
6. **CQI Distribution Analysis** -- Empirical CDF and histograms comparing methods.
7. **Quality Category Summary** -- Grouped bar chart and percentage table.
8. **Component Score Box Plots** -- Box plots comparing components across methods.

In [ ]:
"""
Step 1: Environment Setup
"""
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import logging

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import initialize_config
import src.config as config

initialize_config(os.path.join(project_root, 'config.yml'))

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print(f"Project root: {project_root}")
print(f"Quality path template: {config.quality_path_template}")

# --- User Settings ---
TARGET_MONTH = 1
TARGET_DEKAD = 1

print(f"\nTarget: Month {TARGET_MONTH}, Dekad {TARGET_DEKAD}")

### Step 2: Load QA Results

Load the quality assessment NetCDF files for all three bias correction methods.
Each file contains:
- `basic_statistical_quality` (0--1)
- `distribution_quality` (0--1)
- `temporal_quality` (0--1)
- `continuous_quality` (0--1) -- the CQI
- `categorical_quality` (1--4) -- Poor / Fair / Good / Excellent
- `confidence_level` (0--1)

In [ ]:
"""
Step 2: Load QA NetCDF files for each correction method.
"""
month_str = f"{TARGET_MONTH:02d}"
dekad_str = '01' if TARGET_DEKAD == 1 else ('11' if TARGET_DEKAD == 2 else '21')

methods = {'LS': 'ls', 'LSEQM': 'lseqm', 'LSEQMDL': 'lseqmdl'}
ref_label = 'cpc'

quality_data = {}

for display_name, method_abbr in methods.items():
    quality_dir = config.quality_path_template.replace('{method}', method_abbr)
    test_label = f"imergl_{method_abbr}"
    fname = f"idn_cli_quality_{ref_label}_{test_label}_month{month_str}_dekad{dekad_str}.nc4"
    fpath = os.path.join(quality_dir, fname)

    print(f"\n--- {display_name} ---")
    print(f"Path: {fpath}")

    if os.path.exists(fpath):
        ds = xr.open_dataset(fpath, engine=config.NETCDF_ENGINE)
        quality_data[display_name] = ds
        print(f"Variables: {list(ds.data_vars)}")
        print(f"Dimensions: {dict(ds.dims)}")
    else:
        print(f"WARNING: File not found -- skipping {display_name}")

if not quality_data:
    raise FileNotFoundError(
        "No QA output files found. Run notebook 04_qa_framework first."
    )

print(f"\nLoaded methods: {list(quality_data.keys())}")

### Step 3: CQI Spatial Maps

The **Continuous Quality Index (CQI)** is a weighted composite of basic statistical,
distribution, and temporal quality scores. Values range from 0 (poor) to 1 (excellent).

In [ ]:
"""
Step 3: CQI spatial maps for each method.
"""
n_methods = len(quality_data)
fig, axes = plt.subplots(1, n_methods, figsize=(6 * n_methods, 5), squeeze=False)
axes = axes.flatten()

titles_map = {'LS': 'LS', 'LSEQM': 'LSEQM', 'LSEQMDL': 'LSEQM+DL'}
im = None

for idx, (method_name, ds) in enumerate(quality_data.items()):
    ax = axes[idx]
    cqi = ds['continuous_quality']
    if 'time' in cqi.dims:
        cqi = cqi.isel(time=0)

    im = ax.pcolormesh(
        cqi.lon, cqi.lat, cqi.values,
        cmap='viridis', vmin=0, vmax=1, shading='auto'
    )
    ax.set_xlim(95, 141)
    ax.set_ylim(-11, 6)
    ax.set_title(titles_map.get(method_name, method_name), fontsize=13)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

fig.suptitle(
    f'Continuous Quality Index (CQI)\nMonth {TARGET_MONTH}, Dekad {TARGET_DEKAD}',
    fontsize=14, fontweight='bold', y=1.02
)

if im is not None:
    cbar = fig.colorbar(im, ax=axes.tolist(), orientation='horizontal',
                        fraction=0.05, pad=0.12, aspect=40)
    cbar.set_label('CQI (0 = poor, 1 = excellent)')

plt.tight_layout()
plt.show()

### Step 4: Categorical Quality Maps

| Value | Category  | CQI Range |
|-------|-----------|-----------|
| 1     | Poor      | < 0.4     |
| 2     | Fair      | 0.4 -- 0.6|
| 3     | Good      | 0.6 -- 0.8|
| 4     | Excellent | >= 0.8    |

In [ ]:
"""
Step 4: Categorical quality maps (Poor / Fair / Good / Excellent).
"""
from matplotlib.colors import ListedColormap, BoundaryNorm

cat_colors = ['#d32f2f', '#ff9800', '#9ccc65', '#388e3c']
cat_cmap = ListedColormap(cat_colors)
cat_boundaries = [0.5, 1.5, 2.5, 3.5, 4.5]
cat_norm = BoundaryNorm(cat_boundaries, cat_cmap.N)

n_methods = len(quality_data)
fig, axes = plt.subplots(1, n_methods, figsize=(6 * n_methods, 5), squeeze=False)
axes = axes.flatten()

im = None
for idx, (method_name, ds) in enumerate(quality_data.items()):
    ax = axes[idx]
    cat = ds['categorical_quality']
    if 'time' in cat.dims:
        cat = cat.isel(time=0)
    cat_masked = cat.where(cat > 0)

    im = ax.pcolormesh(
        cat_masked.lon, cat_masked.lat, cat_masked.values,
        cmap=cat_cmap, norm=cat_norm, shading='auto'
    )
    ax.set_xlim(95, 141)
    ax.set_ylim(-11, 6)
    ax.set_title(titles_map.get(method_name, method_name), fontsize=13)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

fig.suptitle(
    f'Categorical Quality Classification\nMonth {TARGET_MONTH}, Dekad {TARGET_DEKAD}',
    fontsize=14, fontweight='bold', y=1.02
)

if im is not None:
    cbar = fig.colorbar(im, ax=axes.tolist(), orientation='horizontal',
                        fraction=0.05, pad=0.12, aspect=40, ticks=[1, 2, 3, 4])
    cbar.ax.set_xticklabels(['Poor', 'Fair', 'Good', 'Excellent'])
    cbar.set_label('Quality Category')

plt.tight_layout()
plt.show()

### Step 5: Method Improvement Map

Quality gain (or loss) from the simplest method (LS) to the most advanced
(LSEQM+DL). Positive values (blue) = improvement, negative (red) = degradation.

In [ ]:
"""
Step 5: Method improvement map (LSEQMDL CQI minus LS CQI).
"""
if 'LS' in quality_data and 'LSEQMDL' in quality_data:
    cqi_ls = quality_data['LS']['continuous_quality']
    cqi_dl = quality_data['LSEQMDL']['continuous_quality']
    if 'time' in cqi_ls.dims:
        cqi_ls = cqi_ls.isel(time=0)
    if 'time' in cqi_dl.dims:
        cqi_dl = cqi_dl.isel(time=0)

    improvement = cqi_dl - cqi_ls

    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.pcolormesh(
        improvement.lon, improvement.lat, improvement.values,
        cmap='RdBu_r', vmin=-0.5, vmax=0.5, shading='auto'
    )
    ax.set_xlim(95, 141)
    ax.set_ylim(-11, 6)
    ax.set_title(
        f'CQI Improvement: LSEQM+DL minus LS\nMonth {TARGET_MONTH}, Dekad {TARGET_DEKAD}',
        fontsize=14, fontweight='bold'
    )
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

    cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.03, pad=0.04)
    cbar.set_label('CQI Difference (positive = improvement)')
    plt.tight_layout()
    plt.show()

    # Summary
    imp_vals = improvement.values[~np.isnan(improvement.values)]
    n_total = len(imp_vals)
    if n_total > 0:
        n_improved = np.sum(imp_vals > 0.001)
        n_degraded = np.sum(imp_vals < -0.001)
        n_unchanged = n_total - n_improved - n_degraded
        print(f"Mean improvement:  {np.mean(imp_vals):.4f}")
        print(f"Median improvement: {np.median(imp_vals):.4f}")
        print(f"Pixels improved:   {n_improved} ({100*n_improved/n_total:.1f}%)")
        print(f"Pixels degraded:   {n_degraded} ({100*n_degraded/n_total:.1f}%)")
        print(f"Pixels unchanged:  {n_unchanged} ({100*n_unchanged/n_total:.1f}%)")
else:
    print('Both LS and LSEQMDL must be loaded for improvement map.')

### Step 6: Component Quality Maps

The CQI is composed of three weighted sub-scores (shown for LSEQM+DL):
- **Basic Statistical Quality** (weight 0.35): RB, RMSE, NSE
- **Distribution Quality** (weight 0.35): Percentile matching, variability, KS test
- **Temporal Quality** (weight 0.30): CSI, event timing, spell preservation

In [ ]:
"""
Step 6: Component quality maps (basic_statistical, distribution, temporal).
"""
comp_method = 'LSEQMDL' if 'LSEQMDL' in quality_data else list(quality_data.keys())[-1]
ds_comp = quality_data[comp_method]

components = [
    ('basic_statistical_quality', 'Basic Statistical Quality'),
    ('distribution_quality', 'Distribution Quality'),
    ('temporal_quality', 'Temporal Quality'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), squeeze=False)
axes = axes.flatten()

for idx, (var_name, var_label) in enumerate(components):
    ax = axes[idx]
    data = ds_comp[var_name]
    if 'time' in data.dims:
        data = data.isel(time=0)

    im = ax.pcolormesh(
        data.lon, data.lat, data.values,
        cmap='viridis', vmin=0, vmax=1, shading='auto'
    )
    ax.set_xlim(95, 141)
    ax.set_ylim(-11, 6)
    ax.set_title(var_label, fontsize=12)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

fig.suptitle(
    f'Quality Components -- {comp_method}\nMonth {TARGET_MONTH}, Dekad {TARGET_DEKAD}',
    fontsize=14, fontweight='bold', y=1.02
)

cbar = fig.colorbar(im, ax=axes.tolist(), orientation='horizontal',
                    fraction=0.05, pad=0.12, aspect=40)
cbar.set_label('Score (0 = poor, 1 = excellent)')
plt.tight_layout()
plt.show()

### Step 7: Confidence Map

The **confidence level** indicates how reliable the quality assessment is at each
pixel. It combines metric consistency, distribution agreement (KS p-value),
and NSE reliability.

In [ ]:
"""
Step 7: Confidence map (LSEQMDL).
"""
conf_method = 'LSEQMDL' if 'LSEQMDL' in quality_data else list(quality_data.keys())[-1]
confidence = quality_data[conf_method]['confidence_level']
if 'time' in confidence.dims:
    confidence = confidence.isel(time=0)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.pcolormesh(
    confidence.lon, confidence.lat, confidence.values,
    cmap='YlOrRd_r', vmin=0, vmax=1, shading='auto'
)
ax.set_xlim(95, 141)
ax.set_ylim(-11, 6)
ax.set_title(
    f'Confidence Level -- {conf_method}\nMonth {TARGET_MONTH}, Dekad {TARGET_DEKAD}',
    fontsize=14, fontweight='bold'
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_aspect('equal')

cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.03, pad=0.04)
cbar.set_label('Confidence (0 = low, 1 = high)')
plt.tight_layout()
plt.show()

conf_vals = confidence.values[~np.isnan(confidence.values)]
if len(conf_vals) > 0:
    print(f"Mean confidence:  {np.mean(conf_vals):.4f}")
    print(f"Min confidence:   {np.min(conf_vals):.4f}")
    print(f"Max confidence:   {np.max(conf_vals):.4f}")

### Step 8: CQI Distribution Analysis

- **Empirical CDF**: what fraction of pixels fall below each CQI value.
- **Histogram**: frequency distribution of CQI.

Vertical dashed lines mark the categorical quality boundaries (0.4, 0.6, 0.8).

In [ ]:
"""
Step 8: CQI distribution plots (CDF + histogram).
"""
method_colors = {'LS': '#1f77b4', 'LSEQM': '#ff7f0e', 'LSEQMDL': '#2ca02c'}

fig, (ax_cdf, ax_hist) = plt.subplots(1, 2, figsize=(14, 5))
cat_thresholds = [0.4, 0.6, 0.8]

for method_name, ds in quality_data.items():
    cqi = ds['continuous_quality']
    if 'time' in cqi.dims:
        cqi = cqi.isel(time=0)
    vals = cqi.values.flatten()
    vals = vals[~np.isnan(vals)]

    color = method_colors.get(method_name, None)
    label = titles_map.get(method_name, method_name)

    # Empirical CDF
    sorted_vals = np.sort(vals)
    ecdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    ax_cdf.plot(sorted_vals, ecdf, label=label, color=color, linewidth=1.5)

    # Histogram
    ax_hist.hist(vals, bins=20, alpha=0.45, label=label, color=color,
                 edgecolor='white', linewidth=0.5, range=(0, 1))

    print(f"Median CQI ({label}): {np.median(vals):.4f}")

# Category boundary lines
for thresh in cat_thresholds:
    ax_cdf.axvline(thresh, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
    ax_hist.axvline(thresh, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)

ax_cdf.set_xlabel('CQI')
ax_cdf.set_ylabel('Cumulative Fraction')
ax_cdf.set_title('Empirical CDF of CQI')
ax_cdf.set_xlim(0, 1)
ax_cdf.set_ylim(0, 1)
ax_cdf.legend()
ax_cdf.grid(True, alpha=0.3)

ax_hist.set_xlabel('CQI')
ax_hist.set_ylabel('Pixel Count')
ax_hist.set_title('Histogram of CQI')
ax_hist.set_xlim(0, 1)
ax_hist.legend()
ax_hist.grid(True, alpha=0.3)

fig.suptitle(
    f'CQI Distribution Analysis -- Month {TARGET_MONTH}, Dekad {TARGET_DEKAD}',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

### Step 9: Quality Category Summary

Grouped bar chart and percentage table showing how many pixels fall into each
quality category for each correction method.

In [ ]:
"""
Step 9: Quality category summary (bar chart + table).
"""
cat_names = ['Poor', 'Fair', 'Good', 'Excellent']
cat_values = [1, 2, 3, 4]
cat_bar_colors = ['#d32f2f', '#ff9800', '#9ccc65', '#388e3c']

# Compute counts per category per method
summary = {}
for method_name, ds in quality_data.items():
    cat = ds['categorical_quality']
    if 'time' in cat.dims:
        cat = cat.isel(time=0)
    vals = cat.values.flatten()
    vals = vals[(vals > 0) & ~np.isnan(vals)]
    total = len(vals)
    counts = [int(np.sum(vals == cv)) for cv in cat_values]
    summary[method_name] = {'counts': counts, 'total': total}

# Grouped bar chart
x = np.arange(len(cat_names))
width = 0.25
n = len(quality_data)

fig, ax = plt.subplots(figsize=(10, 5))
for i, (method_name, info) in enumerate(summary.items()):
    offset = (i - (n - 1) / 2) * width
    pcts = [100 * c / info['total'] if info['total'] > 0 else 0
            for c in info['counts']]
    label = titles_map.get(method_name, method_name)
    ax.bar(x + offset, pcts, width, label=label,
           color=[cat_bar_colors[j] for j in range(len(cat_names))],
           edgecolor='white', linewidth=0.5, alpha=0.7 + 0.1 * i)

ax.set_xlabel('Quality Category')
ax.set_ylabel('Percentage of Pixels (%)')
ax.set_title(
    f'Quality Category Distribution\nMonth {TARGET_MONTH}, Dekad {TARGET_DEKAD}',
    fontsize=14, fontweight='bold'
)
ax.set_xticks(x)
ax.set_xticklabels(cat_names)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Percentage table
print(f"\n{'Method':<12}  {'Poor':>8}  {'Fair':>8}  {'Good':>8}  {'Excellent':>10}  {'Total':>10}")
print('-' * 68)
for method_name, info in summary.items():
    total = info['total']
    pcts = [100 * c / total if total > 0 else 0 for c in info['counts']]
    label = titles_map.get(method_name, method_name)
    print(f"{label:<12}  {pcts[0]:>7.1f}%  {pcts[1]:>7.1f}%  {pcts[2]:>7.1f}%"
          f"  {pcts[3]:>9.1f}%  {total:>10,}")

### Step 10: Component Score Box Plots

Box plots comparing the distribution of each quality component across all
methods. This identifies which aspect benefits most from progressive refinement.

In [ ]:
"""
Step 10: Component score box plots.
"""
try:
    import seaborn as sns
    use_seaborn = True
except ImportError:
    use_seaborn = False
    print('seaborn not available; using matplotlib box plots.')

component_vars = [
    ('basic_statistical_quality', 'Basic Statistical'),
    ('distribution_quality', 'Distribution'),
    ('temporal_quality', 'Temporal'),
    ('continuous_quality', 'CQI'),
]

# Build long-form DataFrame
records = []
for method_name, ds in quality_data.items():
    label = titles_map.get(method_name, method_name)
    for var_name, var_label in component_vars:
        data = ds[var_name]
        if 'time' in data.dims:
            data = data.isel(time=0)
        vals = data.values.flatten()
        vals = vals[~np.isnan(vals)]
        for v in vals:
            records.append({'Method': label, 'Component': var_label, 'Score': float(v)})

df_box = pd.DataFrame(records)

if use_seaborn and len(df_box) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=df_box, x='Component', y='Score', hue='Method',
                ax=ax, palette='Set2', fliersize=1, linewidth=0.8)
    ax.set_ylim(0, 1)
    ax.set_title(
        f'Quality Component Scores by Method\nMonth {TARGET_MONTH}, Dekad {TARGET_DEKAD}',
        fontsize=14, fontweight='bold'
    )
    ax.set_ylabel('Score (0-1)')
    ax.set_xlabel('Quality Component')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(title='Method')
    plt.tight_layout()
    plt.show()

elif len(df_box) > 0:
    method_order = [titles_map.get(m, m) for m in quality_data.keys()]
    n_comp = len(component_vars)
    fig, axes = plt.subplots(1, n_comp, figsize=(4 * n_comp, 6), squeeze=False)
    axes = axes.flatten()
    for c_idx, (var_name, var_label) in enumerate(component_vars):
        ax = axes[c_idx]
        bp_data = [df_box[(df_box['Component'] == var_label) &
                          (df_box['Method'] == m)]['Score'].values
                   for m in method_order]
        ax.boxplot(bp_data, labels=method_order, patch_artist=True)
        ax.set_ylim(0, 1)
        ax.set_title(var_label)
        ax.set_ylabel('Score')
        ax.grid(True, axis='y', alpha=0.3)
    fig.suptitle(
        f'Quality Component Scores by Method\nMonth {TARGET_MONTH}, Dekad {TARGET_DEKAD}',
        fontsize=14, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()

else:
    print('No data available for box plots.')